In [1]:
import json
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import re
import torch
from transformers import BertTokenizer, BertModel
import plotly.graph_objects as go
import numpy as np
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.manifold import MDS
from umap.umap_ import UMAP

In [2]:
# embed all the sentences
tokenizer = BertTokenizer.from_pretrained('l3cube-pune/kannada-bert') #l3cube-pune/kannada-bert
model = BertModel.from_pretrained('l3cube-pune/kannada-bert')

Some weights of BertModel were not initialized from the model checkpoint at l3cube-pune/kannada-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
with open('processed.json', encoding="utf-8") as file:
    data = json.load(file)

keys = data.keys()

In [4]:
words = list(set(keys))
# sort
words.sort()

In [5]:
def get_sentences(file_path, word):
    matching_sentences = []
    word_pattern = re.compile(r'\b{}\b'.format(re.escape(word)))  # Pre-compile the regex pattern for the word
    with open(file_path, encoding='utf-8') as f:
        for line in f:
            # This assumes that sentences do not span multiple lines.
            # You might need a more sophisticated approach if your sentences can span multiple lines.
            potential_sentences = re.split(r'(?<=[.!?])\s+', line)  # Split line into sentences
            for sentence in potential_sentences:
                if word_pattern.search(sentence):  # Check if the sentence contains the word
                    matching_sentences.append(sentence)
    return matching_sentences

# path to my text file
file_path = 'sentences.txt'




In [6]:
sns = get_sentences('sentences.txt','ಅಡಿ')

In [7]:
sns

[' 66000 ಚದರ ಅಡಿಗಳ ವಿಸ್ತೀರ್ಣದಲ್ಲಿ ಹರಡಿಕೊಂಡಿರುವ ಇವರ 60 ಅಡಿಯ ಒಂದು ಈಜುಕೊಳ 2500 ಚದರ ಅಡಿ ವ್ಯಾಯಾಮ ಕೊಠಡಿ ಮತ್ತು 1000 ಚದರ ಅಡಿಗಳ ಊಟದ ಕೊಠಡಿಗಳನ್ನು ಒಳಗೊಂಡಿದೆ\n',
 ' ಕಡೆ 30–40 ಅಡಿ ಅಗಲಕ್ಕೆ ರಸ್ತೆ ವಿಸ್ತರಣೆ ಆಗುತ್ತಿದ್ದರೆ ಇನ್ನೂ ಕೆಲವು ಕಡೆ 15–20 ಅಡಿಗೆ ಇಳಿದಿದೆ ಹೊರವರ್ತುಲ ರಸ್ತೆ ನಿರ್ಮಾಣದ ನೆಪವೊಡ್ಡಿ ನಗರದ ಒಳಗೆ ಭೂಸ್ವಾಧೀನ ಪ್ರಕ್ರಿಯೆಯನ್ನು ಕೈಬಿಡಲಾಗಿದೆ’ ಎಂದು ಸ್ಥಳೀಯರಾದ ರವಿಶಂಕರ್ ಆರೋಪಿಸುತ್ತಾರೆ\n',
 'ಇಂದಿಗೂ ಸುಸ್ಥಿತಿ ಯಲ್ಲಿರುವ ಗೋಳಾಕಾರದ ದೊಡ್ಡಕಟ್ಟಡಗಳಲ್ಲಿ ಪಾಂಟಿಯನ್ ಸ್ವರ್ಣ ದೇವತೆಗಳು ಕಟ್ಟಡ ವೃತ್ತಾಕಾರವಾಗಿದ್ದ ಕೊರಿಂಥಿಯನ್ ಸ್ತಂಭ 20 ಅಡಿಗೂ ಹೆಚ್ಚು ದಪ್ಪವಾದ ಗೋಡೆ 140 ಅಡಿ ಎತ್ತರದ ಬೃಹತ್ ಗುಮ್ಮಟಗಳನ್ನು ಹೊಂದಿದೆ\n',
 '5060 ಅಡಿ ಎತ್ತರದ ಅಡಿಕೆ ಮರದ ತುದಿಯಿಂದ ಸಂಭ್ರಮದಿಂದಲೇ ಇಳಿದು ಬಂದ ಅಪ್ಪ ಜಗತ್ತನ್ನು ಗೆದ್ದಷ್ಟು ಸಂತಸದಿಂದ ನನ್ನನ್ನು ನೋಡಿದ್ದರು\n',
 'ಸರ್ ಅಂತ 12 ಅಡಿ ಜಾರಿ ನನ್ನ ಅಡಿಗಳು ಆ ದಿಂಡಿಗೆ ಹೊಡೆದವು\n',
 'ಇಲ್ಲಿನ ಅಶೋಕ ವೃತ್ತದಲ್ಲಿ ಸೋಮ\xadವಾರ ಮುಖ್ಯಮಂತ್ರಿಗಳ ನಗರೋತ್ಥಾನ ಯೋಜನೆಯ ಸಣ್ಣ ಅತೀ ಸಣ್ಣ ಮತ್ತು ದೊಡ್ಡ ನಗರಗಳನ್ನು ಅಭಿವೃದ್ಧಿ ಪಡಿ\xadಸುವ ಯೋಜನೆ ಅಡಿಯಲ್ಲಿ ನಗರಕ್ಕೆ ಬಿಡುಗಡೆಯಾದ ರೂ\n',
 'ಅಂದ ಹಾಗೆ ಹೇಗಿದೆ ವಿಜಯ ಕರ್ನಾಟಕ ಮಂಗಳೂರು ವಿಭಾಗ ನಿಮ್ಮ ಆಫೀಸಿನ ಎಲ್ಲರಿಗೂ ನನ್ನ ಹಾರೈಕೆಗಳನ್ನು ತಿಳಿಸಿ  ಬಿಳಿ ಹಾಳೆ ಮತ್

In [8]:
def embed_sentence(sentence):
    input_ids = tokenizer.encode(sentence, return_tensors='pt')
    with torch.no_grad():
        model_output = model(input_ids)

    return model_output.last_hidden_state[:, 0, :]

import torch

def embed_word_in_sentence(sentence, word):
    # Tokenize the sentence and find the token ids
    input_ids = tokenizer.encode(sentence, return_tensors='pt')
    
    # Tokenize the word separately to find its token id(s)
    word_ids = tokenizer.encode(word, add_special_tokens=False)
    
    with torch.no_grad():
        model_output = model(input_ids)
    
    # Find all occurrences of the word tokens in the input ids
    word_token_positions = [i for i, input_id in enumerate(input_ids[0]) if input_id in word_ids]
    
    # Extract the embeddings for the word tokens
    # Note: This extracts all pieces of the word if it's split into subwords
    word_embeddings = model_output.last_hidden_state[:, word_token_positions, :]
    
    # Option 1: Return all embeddings for the word (in case it's split into multiple tokens)
    # return word_embeddings

    # Option 2: If you want to aggregate the embeddings (e.g., by averaging) when the word is tokenized into multiple pieces
    return word_embeddings.mean(dim=1)




In [9]:
embed_word_in_sentence( '150000 ವರೆಗೆ ಸೆಕ್ಷನ್\u200c 80ಸಿ ಅಡಿಯಲ್ಲಿ ಹಾಗೂ ಇದರ ಮೇಲಿನ ಬಡ್ಡಿಯು ಸಂಪೂರ್ಣವಾಗಿ ಸೆಕ್ಷನ್\u200c 10ರ ಅನ್ವಯ ಕರ ರಹಿತ\n', 'ಕರ') 

tensor([[-4.5666e-03, -5.6288e-03,  1.9139e-03,  2.3053e-03,  2.8776e-03,
          4.8575e-03,  6.8964e-03, -6.5559e-03,  5.5465e-03, -3.4937e-04,
         -8.4899e-04, -1.6430e-02,  9.9246e-04, -6.0352e-03, -1.2236e-02,
         -4.4568e-03,  2.1565e-03, -4.6358e-03,  3.0259e-03, -4.0096e-03,
         -8.7064e-03, -1.2859e-02, -8.5630e-03, -5.6867e-03, -1.0677e-03,
          1.4777e-03,  1.2410e-02, -1.0582e-03,  4.6428e-03,  3.0877e-02,
          7.5469e-03,  2.3677e-03, -1.8178e-02, -1.5530e-02, -1.1280e-03,
         -7.9744e-03, -8.8250e-03, -6.4634e-03, -1.0783e-02,  8.0466e-04,
          2.5866e-03,  6.3849e-03, -8.9535e-04, -1.2175e-05, -2.1504e-03,
         -2.9142e-03,  9.2361e-03,  1.5906e-02,  4.9106e-04, -1.3414e-02,
          4.4242e-03, -6.7652e-03,  1.1324e-03, -9.2677e-03,  5.0223e-03,
         -8.5308e-03,  8.4585e-04,  7.5555e-05, -1.1570e-02,  1.2651e-02,
          6.6561e-04, -4.2617e-03, -6.6158e-03,  6.7261e-03,  8.7880e-03,
          3.7207e-03, -4.1178e-03,  5.

In [10]:
def project_group_and_scatter_plot_embeddings(embeddings, sentences, n_clusters=5, dim_reducer='mds'):
    # Step 1: Cluster the embeddings
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(embeddings)
    labels = kmeans.labels_

    # Step 2: Reduce dimensionality to 2D for visualization (using MDS in this case)

    if dim_reducer == 'umap':
        reducer = UMAP(n_components=2, random_state=42)
    elif dim_reducer == 'mds':
        reducer = MDS(n_components=2, random_state=42)
    elif dim_reducer == 'tsne':
        reducer = TSNE(n_components=2, random_state=42)

    embeddings_2d = reducer.fit_transform(embeddings)

    # Step 2: Reduce dimensionality to 2D for visualization (using UMAP in this case)
    umap_reducer = UMAP(n_components=2, random_state=42)
    embeddings_2d = umap_reducer.fit_transform(embeddings)

    # Extract the 2D x and y coordinates
    x = embeddings_2d[:, 0]
    y = embeddings_2d[:, 1]

    # Create scatter plot with Plotly
    fig = go.Figure()

    # Loop through the clusters to plot each one as a separate scatter trace
    for i in range(n_clusters):
        cluster_idx = labels == i
        # get index of sentences in original sentences list
        sentences_ids = [i for i, sentence in enumerate(sentences) if sentence in sentences]

        hover_text = [f"({sent_id}) - {sentences[sent_id]}" for idx, sent_id in zip(cluster_idx, sentences_ids)]
        fig.add_trace(go.Scatter(x=x[cluster_idx], y=y[cluster_idx], mode='markers', 
                                marker=dict(size=10),
                                text=hover_text,  # Sentences for hover
                                hoverinfo="text",
                                name=f'Cluster {i+1}'))

    # Customize layout
    fig.update_layout(title=f'Interactive Scatterplot with Clustered Sentences using {dim_reducer.upper()}',
                      xaxis_title='Dimension 1',
                      yaxis_title='Dimension 2',
                      hovermode='closest',# Tooltip shows info of the closest point
                      width=600,
                      height=600,
                      autosize=False
                      )  
    # Show plot
    fig.show()



In [11]:

def scatter_plot_word_sentences(text, word, n_clusters=5, dim_reducer='mds'):
    print('Getting sentences')
    sentences = get_sentences(text, word)
    sentences = [sentence.split('\t')[1] for sentence in sentences if len(sentence.split('\t')) > 1]

    print('Embedding sentences')
    embeddings = [embed_word_in_sentence(sentence, word) for sentence in sentences]
    embeddings = torch.cat(embeddings)

    print('Projecting and plotting')
    project_group_and_scatter_plot_embeddings(embeddings, sentences, n_clusters=n_clusters, dim_reducer=dim_reducer)

    return sentences




In [12]:
def embed_word_in_sentence(sentence, word):
    # Tokenize the sentence and find the token ids
    input_ids = tokenizer.encode(sentence, return_tensors='pt')
    
    # Tokenize the word separately to find its token id(s)
    word_ids = tokenizer.encode(word, add_special_tokens=False)
    
    with torch.no_grad():
        model_output = model(input_ids)
    
    # Find all occurrences of the word tokens in the input ids
    word_token_positions = [i for i, input_id in enumerate(input_ids[0]) if input_id in word_ids]
    
    # Extract the embeddings for the word tokens
    if not word_token_positions:  # Check if the word is actually found in the sentence
        return None  # Return None or an appropriate value if the word is not found

    word_embeddings = model_output.last_hidden_state[:, word_token_positions, :]

    # Return the mean of embeddings if multiple subwords, else just return the embedding
    return word_embeddings.mean(dim=1) if word_embeddings.size(1) > 1 else word_embeddings.squeeze(1)

def scatter_plot_word_sentences(text, word, n_clusters=5, dim_reducer='mds'):
    print('Getting sentences')
    sentences = get_sentences(text, word)
    print('Embedding sentences')
    embeddings = [embed_word_in_sentence(sentence, word) for sentence in sentences if sentence]
    embeddings = [emb for emb in embeddings if emb is not None]  # Filter out None

    if not embeddings:
        print("No embeddings generated; the word may not be present in any sentences.")
        return []

    embeddings = torch.cat(embeddings)

    print('Projecting and plotting')
    project_group_and_scatter_plot_embeddings(embeddings, sentences, n_clusters=n_clusters, dim_reducer=dim_reducer)

    return sentences


In [13]:
sentences = scatter_plot_word_sentences(file_path, 'ಅಡಿ', n_clusters=5, dim_reducer='mds')

Getting sentences
Embedding sentences
Projecting and plotting


c:\Users\ADMIN\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\ADMIN\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\manifold\_mds.py:298: FutureWarning: The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.
  warnings.warn(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python39\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
c:\Users\ADMIN\AppData\Local\Programs\Python\Python39\lib\site-packages\umap\umap_.py:2437: UserWarning: n_nei

In [14]:
def embed_word_in_sentence(sentence, word, tokenizer, model):
    # Truncate the sentence to the model's max length
    max_length = tokenizer.model_max_length
    
    # Tokenize the sentence and find the token ids
    input_ids = tokenizer.encode(sentence, return_tensors='pt', max_length=max_length, truncation=True)
    
    # Tokenize the word separately to find its token id(s)
    word_ids = tokenizer.encode(word, add_special_tokens=False)
    
    with torch.no_grad():
        model_output = model(input_ids)
    
    # Find all occurrences of the word tokens in the input ids
    word_token_positions = [i for i, id in enumerate(input_ids[0]) if id in word_ids]
    
    # Check if the word is found in the tokenized sentence
    if not word_token_positions:
        return None  # Return None if the word is not present

    # Extract the embeddings for the word tokens
    word_embeddings = model_output.last_hidden_state[:, word_token_positions, :]
    
    # Return the mean of embeddings if there are multiple tokens for the word
    return torch.mean(word_embeddings, dim=1)

# Usage in context
tokenizer = BertTokenizer.from_pretrained('l3cube-pune/kannada-bert')
model = BertModel.from_pretrained('l3cube-pune/kannada-bert')

sentence = "150000 ವರೆಗೆ ಸೆಕ್ಷನ್\u200c 80ಸಿ ಅಡಿಯಲ್ಲಿ ಹಾಗೂ ಇದರ ಮೇಲಿನ ಬಡ್ಡಿಯು ಸಂಪೂರ್ಣವಾಗಿ ಸೆಕ್ಷನ್\u200c 10ರ ಅನ್ವಯ ಕರ ರಹಿತ\n"
word = "ಕರ"
embeddings = embed_word_in_sentence(sentence, word, tokenizer, model)
if embeddings is not None:
    print("Embedding:", embeddings)
else:
    print("Word not found in the sentence.")


Some weights of BertModel were not initialized from the model checkpoint at l3cube-pune/kannada-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Embedding: tensor([[-4.5666e-03, -5.6288e-03,  1.9139e-03,  2.3053e-03,  2.8776e-03,
          4.8575e-03,  6.8964e-03, -6.5559e-03,  5.5465e-03, -3.4937e-04,
         -8.4899e-04, -1.6430e-02,  9.9246e-04, -6.0352e-03, -1.2236e-02,
         -4.4568e-03,  2.1565e-03, -4.6358e-03,  3.0259e-03, -4.0096e-03,
         -8.7064e-03, -1.2859e-02, -8.5630e-03, -5.6867e-03, -1.0677e-03,
          1.4777e-03,  1.2410e-02, -1.0582e-03,  4.6428e-03,  3.0877e-02,
          7.5469e-03,  2.3677e-03, -1.8178e-02, -1.5530e-02, -1.1280e-03,
         -7.9744e-03, -8.8250e-03, -6.4634e-03, -1.0783e-02,  8.0466e-04,
          2.5866e-03,  6.3849e-03, -8.9535e-04, -1.2175e-05, -2.1504e-03,
         -2.9142e-03,  9.2361e-03,  1.5906e-02,  4.9106e-04, -1.3414e-02,
          4.4242e-03, -6.7652e-03,  1.1324e-03, -9.2677e-03,  5.0223e-03,
         -8.5308e-03,  8.4585e-04,  7.5555e-05, -1.1570e-02,  1.2651e-02,
          6.6561e-04, -4.2617e-03, -6.6158e-03,  6.7261e-03,  8.7880e-03,
          3.7207e-03, -4.11

In [15]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt



In [ ]:
num_samples = len(embeddings)
n_clusters = min(num_samples, 5)  # Ensure we do not exceed the number of samples

if num_samples > 1:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(embeddings)

    # Proceed with other operations such as dimensionality reduction and visualization
else:
    print("Not enough data to perform clustering.")